# 05_step4_F2_applied_all_RCM - Apply F2 to all RCM runs (RCM domain, 0.25 deg)

The **same** F2 function (parameters from notebook step3, fixed WDF included) is applied to:
RCM 2025 no-hosing, 2025 hosing, 2050 no-hosing, 2050 hosing.

Nothing is recalibrated on the hosing runs: the RCM response to hosing goes through the correction.
The "hosing - no-hosing" difference (the thesis result) is then computed on these adjusted outputs.

Output: `rcm_ba/<run>_F2.nc` (variables `pr`, `tasmax`, `tasmin`), ready for LISFLOOD / ECroPS.

In [ ]:
import sys, warnings
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent):   # ba_utils.py: same folder as the notebook, or the parent folder
    sys.path.insert(0, str(_p))
import numpy as np, xarray as xr, pandas as pd
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                      # ['pr', 'tasmax', 'tasmin']

APPLY_TO = ["rcm_2025_nohosing", "rcm_2025_hosing", "rcm_2050_nohosing", "rcm_2050_hosing"]
CHUNKS_SPACE = {"lat": 30, "lon": 60}
OVERWRITE = False
F2_DIR = bu.out_dir(cfg, "plan_B", "F2_BA")
RCMBA_DIR = bu.out_dir(cfg, "plan_B", "rcm_ba")
assert (F2_DIR / "qdm_pr.nc").exists(), "Run notebook step3 first (F2 not calibrated)."

## 1. Application

In [ ]:
for name in APPLY_TO:
    out_path = RCMBA_DIR / f"{name}_F2.nc"
    if out_path.exists() and not OVERWRITE:
        print(f"[already done] {name}"); continue
    if not all(bu.dataset_available(cfg, name, v) for v in VARS):
        print(f"[missing] {name} - fill in the paths in notebook 00"); continue
    print(f"-> {name}")
    sim = {v: bu.read_raw(cfg, name, v) for v in VARS}
    adj = bu.apply_F(sim, F2_DIR, chunks_space=CHUNKS_SPACE)
    bu.save_ds(xr.Dataset(adj), out_path)

## 2. Quick checks

Physics (pr >= 0, tasmin < tasmax, no NaN) and shape of the "hosing - no-hosing" signal before / after correction
(monthly means over the RCM domain). In-depth tests (value range, two halves) are in the validations notebook.

In [ ]:
rows = []
for name in APPLY_TO:
    f = RCMBA_DIR / f"{name}_F2.nc"
    if not f.exists():
        continue
    s = xr.open_dataset(f, chunks={"time": 365}).isel(time=slice(0, 365 * 3))
    rows.append({"run": name, "pr min": float(s["pr"].min()),
                 "frac tasmin>=tasmax (should be 0)": float(((s["tasmin"] >= s["tasmax"]) & s["tasmax"].notnull()).mean()),
                 "frac NaN (pr)": float(s["pr"].isnull().mean())})
pd.DataFrame(rows)

In [ ]:
def monthly(da):
    return bu.area_mean(da).groupby("time.month").mean().compute()

fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for r, (a, b) in enumerate([("rcm_2025_hosing", "rcm_2025_nohosing"), ("rcm_2050_hosing", "rcm_2050_nohosing")]):
    fa, fb = RCMBA_DIR / f"{a}_F2.nc", RCMBA_DIR / f"{b}_F2.nc"
    for c, var in enumerate(VARS):
        ax = axes[r, c]
        if not (fa.exists() and fb.exists() and bu.dataset_available(cfg, a, var) and bu.dataset_available(cfg, b, var)):
            ax.set_title(f"{a}-{b} / {var}: missing"); continue
        raw = monthly(bu.read_raw(cfg, a, var)) - monthly(bu.read_raw(cfg, b, var))
        adj = monthly(bu.read_processed(fa, var)) - monthly(bu.read_processed(fb, var))
        ax.plot(raw.month, raw, "o-", label="raw"); ax.plot(adj.month, adj, "s--", label="after F2")
        ax.axhline(0, color="grey", lw=0.5); ax.set_title(f"{a} - {b} : {var}"); ax.set_xlabel("month")
axes[0, 0].legend()
plt.tight_layout(); plt.savefig(F2_DIR / "hosing_signal_before_after.png", dpi=110); plt.show()